# 11. Encoder Comparison: LSTM vs GRU vs Transformer

**Purpose**: Justify the choice of LSTM for the PredictionAgent by comparing
against GRU and Transformer encoders on identical data.

**Key argument**: LSTM offers comparable accuracy to Transformer but with O(1)
incremental inference — critical for the continuous pipeline.

**Tables produced:**
- Table: Encoder comparison (AUC-ROC, AUC-PR, F1, training time, inference time)
- Table: Horizon ablation (AUC-ROC at horizon={1,3,5,10,20})

In [ ]:
import sys, os, time, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from scipy.stats import wilcoxon

ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from agents.utils import set_global_seed, load_config
from agents.prediction_agent import (
    GapSequenceDataset, create_model,
    DEVICE, NUM_TAGS, SEQ_LEN, HORIZON,
    NUM_WORKERS,
)

warnings.filterwarnings('ignore')
config = load_config('configs/config.yaml')
SEEDS = config['global']['seeds_for_repeats']
RESULTS_DIR = Path('results/encoder_comparison')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}, Seeds: {SEEDS}')

## 1. Load Data

In [ ]:
train_df = pd.read_parquet('data/splits/train.parquet')
val_df = pd.read_parquet('data/splits/val.parquet')
test_df = pd.read_parquet('data/splits/test.parquet')

# Build datasets once (deterministic)
train_dataset = GapSequenceDataset(train_df)
val_dataset = GapSequenceDataset(val_df)
test_dataset = GapSequenceDataset(test_df)

print(f'Train: {len(train_dataset)} seqs, Val: {len(val_dataset)}, Test: {len(test_dataset)}')

## 2. Train & Evaluate All Encoders (5 seeds)

In [ ]:
import torch.nn as nn

def train_and_evaluate(model_type, train_ds, val_ds, test_ds, seed,
                       epochs=30, batch_size=512, patience=5):
    """Train one model and return metrics + timing."""
    set_global_seed(seed)
    
    model = create_model(model_type).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    
    # LR: lower for transformer
    lr = 5e-4 if model_type == 'transformer' else 1e-3
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    
    # ── Train ──
    t_train_start = time.time()
    best_val_loss = float('inf')
    best_state = None
    no_improve = 0
    
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        
        # Validate
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                val_losses.append(criterion(model(xb), yb).item())
        
        vl = np.mean(val_losses)
        if vl < best_val_loss:
            best_val_loss = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
    
    train_time = time.time() - t_train_start
    model.load_state_dict(best_state)
    model.eval()
    
    # ── Test metrics ──
    all_preds, all_labels = [], []
    t_infer_start = time.time()
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE)
            all_preds.append(torch.sigmoid(model(xb)).cpu().numpy())
            all_labels.append(yb.numpy())
    infer_time = time.time() - t_infer_start
    
    y_true = np.concatenate(all_labels)
    y_score = np.concatenate(all_preds)
    mask = y_true.sum(axis=0) > 0
    
    try:
        auc_roc = roc_auc_score(y_true[:, mask], y_score[:, mask], average='macro')
    except ValueError:
        auc_roc = 0.0
    try:
        auc_pr = average_precision_score(y_true[:, mask], y_score[:, mask], average='macro')
    except ValueError:
        auc_pr = 0.0
    
    y_pred_bin = (y_score >= 0.5).astype(int)
    f1 = f1_score(y_true[:, mask], y_pred_bin[:, mask], average='macro', zero_division=0)
    
    # ── Per-batch inference timing ──
    single_batch = next(iter(test_loader))[0][:32].to(DEVICE)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    t0 = time.perf_counter()
    for _ in range(100):
        with torch.no_grad():
            model(single_batch)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    batch_infer_ms = (time.perf_counter() - t0) / 100 * 1000
    
    return {
        'model_type': model_type,
        'seed': seed,
        'auc_roc': round(auc_roc, 4),
        'auc_pr': round(auc_pr, 4),
        'f1_macro': round(f1, 4),
        'train_time_s': round(train_time, 1),
        'infer_time_s': round(infer_time, 3),
        'batch_infer_ms': round(batch_infer_ms, 2),
        'n_params': n_params,
        'best_epoch': epoch + 1 - no_improve,
    }

In [ ]:
%%time

all_results = []

for model_type in ['lstm', 'gru', 'transformer']:
    print(f'\n{"="*50}')
    print(f'Training {model_type.upper()}')
    print(f'{"="*50}')
    
    for seed in SEEDS:
        r = train_and_evaluate(
            model_type, train_dataset, val_dataset, test_dataset, seed
        )
        all_results.append(r)
        print(f'  seed={seed}: AUC-ROC={r["auc_roc"]:.4f}, '
              f'AUC-PR={r["auc_pr"]:.4f}, F1={r["f1_macro"]:.4f}, '
              f'train={r["train_time_s"]:.0f}s, batch={r["batch_infer_ms"]:.1f}ms')

results_df = pd.DataFrame(all_results)
results_df.to_csv(RESULTS_DIR / 'encoder_raw_results.csv', index=False)
print(f'\nSaved raw results: {len(results_df)} rows')

## 3. Summary Table (for paper)

In [ ]:
# Aggregate: mean +/- std per model
summary_rows = []
for model_type in ['lstm', 'gru', 'transformer']:
    df_m = results_df[results_df['model_type'] == model_type]
    row = {'Encoder': model_type.upper()}
    for metric in ['auc_roc', 'auc_pr', 'f1_macro', 'train_time_s', 'batch_infer_ms']:
        vals = df_m[metric].values
        mean, std = np.mean(vals), np.std(vals, ddof=1)
        if metric in ('train_time_s', 'batch_infer_ms'):
            row[metric] = f'{mean:.1f}\u00b1{std:.1f}'
        else:
            row[metric] = f'{mean:.4f}\u00b1{std:.4f}'
    row['n_params'] = f'{df_m["n_params"].iloc[0]:,}'
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.columns = ['Encoder', 'AUC-ROC', 'AUC-PR', 'F1-macro',
                    'Train (s)', 'Infer/batch (ms)', 'Parameters']

print('Table: Encoder Comparison (mean \u00b1 std over 5 seeds)')
print('=' * 100)
print(summary.to_string(index=False))

summary.to_csv(RESULTS_DIR / 'encoder_comparison_table.csv', index=False)

In [ ]:
# Statistical test: LSTM vs Transformer
lstm_auc = results_df[results_df['model_type'] == 'lstm']['auc_roc'].values
transformer_auc = results_df[results_df['model_type'] == 'transformer']['auc_roc'].values
gru_auc = results_df[results_df['model_type'] == 'gru']['auc_roc'].values

if len(lstm_auc) >= 5 and len(transformer_auc) >= 5:
    stat_lt, p_lt = wilcoxon(lstm_auc, transformer_auc)
    stat_lg, p_lg = wilcoxon(lstm_auc, gru_auc)
    
    print(f'LSTM vs Transformer (AUC-ROC): p={p_lt:.4f} {"*" if p_lt < 0.05 else "ns"}')
    print(f'LSTM vs GRU (AUC-ROC): p={p_lg:.4f} {"*" if p_lg < 0.05 else "ns"}')
    print()
    
    # Speed comparison
    lstm_speed = results_df[results_df['model_type'] == 'lstm']['batch_infer_ms'].mean()
    trans_speed = results_df[results_df['model_type'] == 'transformer']['batch_infer_ms'].mean()
    speedup = trans_speed / lstm_speed if lstm_speed > 0 else 0
    
    print(f'Inference speed: LSTM={lstm_speed:.1f}ms, Transformer={trans_speed:.1f}ms')
    print(f'LSTM is {speedup:.1f}x faster per batch')
    print()
    print('Paper statement:')
    print(f'  "LSTM was chosen for its O(1) incremental update capability, '
          f'critical for the continuous pipeline. Transformer achieves '
          f'comparable AUC-ROC ({np.mean(transformer_auc):.3f} vs {np.mean(lstm_auc):.3f}, '
          f'p = {p_lt:.2f}) but requires full sequence re-computation, '
          f'making it {speedup:.0f}x slower for online inference."')
else:
    print('Not enough results for statistical test. Ensure all 5 seeds completed.')

## 4. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, title in zip(axes,
    ['auc_roc', 'auc_pr', 'batch_infer_ms'],
    ['AUC-ROC (higher=better)', 'AUC-PR (higher=better)', 'Inference Time ms (lower=better)']):
    
    data = [results_df[results_df['model_type'] == m][metric].values
            for m in ['lstm', 'gru', 'transformer']]
    
    bp = ax.boxplot(data, labels=['LSTM', 'GRU', 'Transformer'],
                    patch_artist=True)
    colors = ['#2196F3', '#4CAF50', '#FF9800']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Encoder Comparison (5 seeds)', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'encoder_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Horizon Ablation

Test AUC-ROC across different prediction horizons: {1, 3, 5, 10, 20}.

In [ ]:
%%time

horizons = [1, 3, 5, 10, 20]
horizon_results = []

for horizon in horizons:
    print(f'\nHorizon={horizon}:')
    
    # Rebuild dataset with different horizon
    train_ds_h = GapSequenceDataset(train_df, horizon=horizon)
    val_ds_h = GapSequenceDataset(val_df, horizon=horizon)
    test_ds_h = GapSequenceDataset(test_df, horizon=horizon)
    
    if len(train_ds_h) == 0 or len(test_ds_h) == 0:
        print(f'  Skipping: not enough sequences')
        continue
    
    for model_type in ['lstm', 'transformer']:
        # Use seed=42 only for speed (full multi-seed for main comparison)
        r = train_and_evaluate(
            model_type, train_ds_h, val_ds_h, test_ds_h,
            seed=42, epochs=20
        )
        r['horizon'] = horizon
        horizon_results.append(r)
        print(f'  {model_type.upper()}: AUC-ROC={r["auc_roc"]:.4f}')

horizon_df = pd.DataFrame(horizon_results)
horizon_df.to_csv(RESULTS_DIR / 'horizon_ablation.csv', index=False)

In [ ]:
# Horizon ablation table
if len(horizon_df) > 0:
    pivot = horizon_df.pivot(index='horizon', columns='model_type', values='auc_roc')
    print('Table: Horizon Ablation (AUC-ROC)')
    print('=' * 50)
    print(pivot.to_string())
    
    # Plot
    fig, ax = plt.subplots(figsize=(8, 5))
    for model_type, color, marker in [('lstm', '#2196F3', 'o'),
                                       ('transformer', '#FF9800', 's')]:
        data = horizon_df[horizon_df['model_type'] == model_type]
        if len(data) > 0:
            ax.plot(data['horizon'], data['auc_roc'],
                    f'-{marker}', color=color, linewidth=2, markersize=8,
                    label=model_type.upper())
    
    ax.set_xlabel('Prediction Horizon (next N questions)')
    ax.set_ylabel('AUC-ROC')
    ax.set_title('AUC-ROC vs Prediction Horizon')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xticks(horizons)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'horizon_ablation.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No horizon results available.')

## 6. Incremental Update Timing

LSTM/GRU: update hidden state with one new interaction = O(1)  
Transformer: re-process entire sequence = O(T)

In [ ]:
# Simulate incremental inference
set_global_seed(42)

seq = torch.randn(1, SEQ_LEN, 6).to(DEVICE)
n_runs = 1000

incremental_times = {}

# LSTM: feed one timestep with hidden state
lstm_model = create_model('lstm').to(DEVICE).eval()
with torch.no_grad():
    # Warm up and get hidden state from sequence
    _ = lstm_model(seq)
    
    # Time: full re-computation (baseline)
    t0 = time.perf_counter()
    for _ in range(n_runs):
        lstm_model(seq)
    full_time_lstm = (time.perf_counter() - t0) / n_runs * 1000
    incremental_times['LSTM (full seq)'] = round(full_time_lstm, 3)

# GRU: same pattern
gru_model = create_model('gru').to(DEVICE).eval()
with torch.no_grad():
    t0 = time.perf_counter()
    for _ in range(n_runs):
        gru_model(seq)
    full_time_gru = (time.perf_counter() - t0) / n_runs * 1000
    incremental_times['GRU (full seq)'] = round(full_time_gru, 3)

# Transformer: must always do full re-computation
trans_model = create_model('transformer').to(DEVICE).eval()
with torch.no_grad():
    t0 = time.perf_counter()
    for _ in range(n_runs):
        trans_model(seq)
    full_time_trans = (time.perf_counter() - t0) / n_runs * 1000
    incremental_times['Transformer (full seq)'] = round(full_time_trans, 3)

print('Single-sample inference time (ms, averaged over 1000 runs):')
for name, t in incremental_times.items():
    print(f'  {name:30s}: {t:.3f} ms')

print(f'\nTransformer/LSTM ratio: {full_time_trans/full_time_lstm:.1f}x')
print(f'Transformer/GRU ratio: {full_time_trans/full_time_gru:.1f}x')

# Note: For true incremental update, LSTM/GRU can feed just the new
# timestep and carry over the hidden state. Transformer must always
# reprocess the entire sequence. This makes LSTM/GRU O(1) amortized
# vs O(T) for Transformer in the continuous pipeline.